# UTS Praktikum Machine Learning

## Agro-Environmental Suitability Classification

Notebook ini dipakai untuk:
- exploratory data analysis (EDA)
- quality check data
- preprocessing numerik dan kategorikal
- hyperparameter tuning
- cross-validation
- evaluasi model
- export model ke folder `model`


## Catatan Penting

Notebook ini sudah disesuaikan dengan backend dan frontend project saat ini. Feature set sekarang lebih lengkap dan mencakup fitur numerik serta kategorikal yang lebih sesuai dengan instruksi UTS.


In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")

current_dir = Path.cwd().resolve()
candidate_roots = [current_dir, *current_dir.parents]
PROJECT_ROOT = next(
    (root for root in candidate_roots if (root / "dataset" / "agro_environmental_dataset.csv").exists()),
    current_dir,
)

DATASET_PATH = PROJECT_ROOT / "dataset" / "agro_environmental_dataset.csv"
MODEL_PATH = PROJECT_ROOT / "model" / "pipeline.pkl"
N_JOBS = 1

NUMERIC_FEATURES = [
    "bulk_density",
    "organic_matter_pct",
    "cation_exchange_capacity",
    "salinity_ec",
    "buffering_capacity",
    "soil_moisture_pct",
    "moisture_limit_dry",
    "moisture_limit_wet",
    "soil_temp_c",
    "air_temp_c",
    "light_intensity_par",
    "soil_ph",
    "ph_stress_flag",
    "nitrogen_ppm",
    "phosphorus_ppm",
    "potassium_ppm",
]
CATEGORICAL_FEATURES = [
    "soil_type",
    "moisture_regime",
    "thermal_regime",
    "nutrient_balance",
    "plant_category",
]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "failure_flag"

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline

    PIPELINE_CLASS = ImbPipeline
    smote_available = True
    sampling_note = "SMOTE aktif"
except ImportError:
    PIPELINE_CLASS = SklearnPipeline
    smote_available = False
    sampling_note = "SMOTE tidak aktif karena imbalanced-learn belum terpasang"

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATASET_PATH)
print("Model path:", MODEL_PATH)
print("Sampling note:", sampling_note)
print("Total features:", len(ALL_FEATURES))


## Load Dataset dan Cek Awal

Tahap awal digunakan untuk memahami struktur dataset, jumlah kolom, dan tipe data yang tersedia.

In [ ]:
df = pd.read_csv(DATASET_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)

df.head()


## Quality Check dan EDA Dasar

Bagian ini memeriksa missing values, data duplikat, distribusi target, dan statistik deskriptif fitur utama.

In [ ]:
missing_values = df[ALL_FEATURES + [TARGET]].isna().sum().sort_values(ascending=False)
duplicate_count = df.duplicated().sum()
class_distribution = df[TARGET].value_counts().sort_index()

print("Missing values:\n", missing_values[missing_values > 0] if (missing_values > 0).any() else "Tidak ada missing value pada fitur yang dipakai")
print("\nDuplicate rows:", duplicate_count)
print("\nClass distribution:\n", class_distribution)
print("\nClass ratio:\n", (class_distribution / len(df)).round(4))

df[NUMERIC_FEATURES].describe().T


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.countplot(x=TARGET, data=df, ax=axes[0])
axes[0].set_title("Distribusi Target")

corr = df[NUMERIC_FEATURES + [TARGET]].corr(numeric_only=True)
sns.heatmap(corr, annot=False, cmap="coolwarm", ax=axes[1])
axes[1].set_title("Korelasi Fitur Numerik")

plt.tight_layout()
plt.show()


In [ ]:
for column in CATEGORICAL_FEATURES:
    print(f"{column}: {sorted(df[column].dropna().unique().tolist())}")


## Preprocessing dan Split Data

Kita memakai kombinasi fitur numerik dan kategorikal. Fitur kategorikal akan di-encode dengan `OneHotEncoder`, sedangkan fitur numerik diimputasi dan diskalakan.

In [ ]:
model_df = df[ALL_FEATURES + [TARGET]].dropna().copy()

X = model_df[ALL_FEATURES]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain class distribution:\n", y_train.value_counts(normalize=True).round(4))


## Pipeline Preprocessing

Preprocessor dipisah antara numerik dan kategorikal agar lebih sesuai dengan instruksi UTS yang menyinggung scaling dan encoding.

In [ ]:
numeric_pipeline = SklearnPipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = SklearnPipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("num", numeric_pipeline, NUMERIC_FEATURES),
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
    ]
)


## Hyperparameter Tuning dan Cross-Validation

Random Forest digunakan sebagai ensemble method utama. Jika `SMOTE` tersedia, penanganan imbalance dilakukan setelah preprocessing.

In [ ]:
steps = [("preprocessor", preprocessor)]

if smote_available:
    steps.append(("smote", SMOTE(random_state=42)))

steps.append(
    (
        "classifier",
        RandomForestClassifier(
            random_state=42,
            class_weight="balanced",
            n_jobs=N_JOBS,
        ),
    )
)

pipeline = PIPELINE_CLASS(steps=steps)

param_distributions = {
    "classifier__n_estimators": [100, 150, 200, 300],
    "classifier__max_depth": [None, 8, 12, 16, 20],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2", None],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="f1",
    cv=cv,
    n_jobs=N_JOBS,
    random_state=42,
    verbose=1,
)

search.fit(X_train, y_train)

best_model = search.best_estimator_
print("Best params:")
print(search.best_params_)
print("\nBest CV F1:", round(search.best_score_, 4))


## Evaluasi Model pada Hold-Out Test Set

Model terbaik diuji pada data test agar performa yang dilaporkan tidak hanya berdasarkan hasil tuning.

In [ ]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_prob),
}

print("Evaluation metrics:")
for name, value in metrics.items():
    print(f"- {name}: {value:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Greens")
plt.title("Confusion Matrix - Best Model")
plt.show()


## Cross-Validation Summary pada Model Terbaik

Cross-validation tambahan dipakai untuk memeriksa kestabilan model pada beberapa metrik penting.

In [ ]:
cv_scores = cross_validate(
    best_model,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc",
    },
    n_jobs=N_JOBS,
)

cv_summary = pd.DataFrame(cv_scores).agg(["mean", "std"]).T
cv_summary


## Export Model

Model terbaik disimpan ke folder `model` agar langsung bisa digunakan oleh backend FastAPI.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, MODEL_PATH)
print(f"Model saved to: {MODEL_PATH.resolve()}")


## Kesimpulan Singkat

Notebook ini sekarang memakai feature set yang lebih lengkap dan lebih sesuai dengan instruksi UTS:
- fitur numerik bertambah sehingga kondisi tanah dan lingkungan lebih terwakili
- fitur kategorikal ikut dimasukkan dan di-encode dalam pipeline
- preprocessing backend dan notebook bergerak dalam arah yang sama
- model yang di-export lebih konsisten dengan input form pada aplikasi
